<div dir="rtl">

# 🧠 05 - Chat History & Memory Management in LangChain 0.3+

## ما هو هذا الكراس؟
يشرح هذا الكراس كيفية إدارة **سجل المحادثة والذاكرة (Chat History & State Management)** في تطبيقات الذكاء الاصطناعي باستخدام المعمارية الحديثة `RunnableWithMessageHistory` و `ChatMessageHistory` المدمجة في LangChain.

## الفوائد والمميزات الرئيسية:
- **إيقاف النسيان (Context Retention)**: تذكر الإجابات والأسماء والموضوعات المذكورة في الرسائل السابقة.
- **الفصل بين الجلسات (Session Isolation)**: إدارة محادثات مستخدمين متعددين عبر معرّفات الجلسات الفريدة (`session_id`).
- **الدعم الديناميكي المتعدد**: إرسال المتغيرات وتخصيص اللغة وحقن السجل بسلاسة بواسطة `MessagesPlaceholder`.

</div>

### 1️⃣ تحميل البيئة واستيراد المكتبات الأساسية

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# تحميل المفاتيح والبيئة
load_dotenv(find_dotenv())
DATA_DIR = Path("data") if Path("data").exists() else Path("../../data") if Path("../../data").exists() else Path("../data")

print("✅ تم تحميل المكتبات وتجهيز البيئة بنجاح.")

✅ تم تحميل المكتبات وتجهيز البيئة بنجاح.


/tmp/ipykernel_37744/2280560326.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


### 2️⃣ إنشاء مخزن الجلسات (Session Store)

In [2]:
# قاموس محلي لحفظ تاريخ الرسائل لكل session_id
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """إرجاع أو إنشاء سجل المحادثة الخاص بالجلسة المحددة"""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

print("✅ تم إنشاء دالة إدارة الذاكرة والجلسات بنجاح.")

✅ تم إنشاء دالة إدارة الذاكرة والجلسات بنجاح.


### 3️⃣ بناء قالب التوجيه مع MessagesPlaceholder وسلسلة LCEL

In [3]:
# تهيئة النموذج التوليدي
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

# إنشاء التوجيه مع وسم MessagesPlaceholder لاستقبال سجل الرسائل السابقة تلقائياً
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "أنت مساعد ذكي ومفيد. أجب على الأسئلة بناءً على سجل المحادثة السابق. "
        "أجب دائماً باللغة {language}."
    ),
    MessagesPlaceholder(variable_name="messages"),
    ("human", "{question}"),
])

# بناء السلسلة الأولوية
chain = prompt | llm

# ربط السلسلة بمحرك الذاكرة الممتدة
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="messages"
)

print("✅ تم بناء السلسلة التفاعلية المعززة بالذاكرة الممتدة بنجاح.")

✅ تم بناء السلسلة التفاعلية المعززة بالذاكرة الممتدة بنجاح.


/home/ali/ME/Code/AI/langchain-mastery/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### 4️⃣ اختبار تذكر المعلومات في نفس الجلسة (Session 1)

In [4]:
config_session_1 = {"configurable": {"session_id": "session_ali_1"}}

# 1. التعرّف على المستخدم وإعطاء المعلومة
query_1 = "أهلاً! أنا اسمي علي، وأنا أعمل مهندس ذكاء اصطناعي وأتعلم LangChain."
print(f"👤 المستخدم: {query_1}\n")

try:
    response_1 = with_message_history.invoke(
        {"language": "العربية", "question": query_1},
        config=config_session_1
    )
    print(f"🤖 المساعد:\n{response_1.content}\n")
    print("-" * 50)
    
    # 2. التثبت من الذاكرة بالسؤال عن الاسم والوظيفة
    query_2 = "هل تتذكر ما هو اسمي وما هي وظيفتي؟"
    print(f"👤 المستخدم: {query_2}\n")
    
    response_2 = with_message_history.invoke(
        {"language": "العربية", "question": query_2},
        config=config_session_1
    )
    print(f"🤖 المساعد (باسترجاع الذاكرة):\n{response_2.content}")
except Exception as e:
    print(f"⚠️ خطأ أثناء التشغيل: {e}")

👤 المستخدم: أهلاً! أنا اسمي علي، وأنا أعمل مهندس ذكاء اصطناعي وأتعلم LangChain.

🤖 المساعد:
أهلاً علي! يسعدني أن أساعدك في تعلم LangChain. هل هناك جانب محدد ترغب في استكشافه؟ مثل إعداد الـChain، التكامل مع LLMs، أو بناء تطبيق عملي؟ أخبرني بما تحتاجه وسأقدم لك إرشادات وأمثلة.

--------------------------------------------------
👤 المستخدم: هل تتذكر ما هو اسمي وما هي وظيفتي؟

🤖 المساعد (باسترجاع الذاكرة):
نعم، أتذكر ذلك. اسميك هو علي، وتعمل مهندس ذكاء اصطناعي وتتعلم LangChain.


### 5️⃣ اختبار العزل التام بين الجلسات (Session Isolation)

In [5]:
# استخدام معرّف جلسة جديد تماماً
config_session_2 = {"configurable": {"session_id": "session_user_2"}}

query_isolated = "هل تعرف ما هو اسمي وما هي وظيفتي؟"
print(f"👤 مستخدم في جلسة جديدة: {query_isolated}\n")

try:
    response_isolated = with_message_history.invoke(
        {"language": "العربية", "question": query_isolated},
        config=config_session_2
    )
    print(f"🤖 المساعد (في الجلسة الثانية المعزولة):\n{response_isolated.content}")
except Exception as e:
    print(f"⚠️ خطأ أثناء التشغيل: {e}")

👤 مستخدم في جلسة جديدة: هل تعرف ما هو اسمي وما هي وظيفتي؟

🤖 المساعد (في الجلسة الثانية المعزولة):
عذرًا، لا أملك معلومات عن اسمك أو وظيفتك في هذه المحادثة. هل يمكنك تزويدي بهذه التفاصيل؟


<div dir="rtl">

## 💡 الخلاصة والخطوات التالية:
- تم النجاح في بناء سلسلة تفاعلية تعتمد على `RunnableWithMessageHistory` لحفظ وتتبع الذاكرة.
- تم إثبات الفصل التام بين معرّفات الجلسات (`session_id`).
- الخطوة التالية: تشغيل مشروع الـ Chatbot المستقل بـ Streamlit في مجلد `projects/04_conversational_chatbot`.

</div>